# The rest of the built-in functions

The numbers and strings notebooks covered:
`abs`, `all`, `any`, `ascii`, `bin`, `bool`, `chr`, `complex`, `divmod`, `enumerate`, `filter`, `float`, `format`, `hex`, `input`, `int`, `isinstance`, `len`, `map`, `max`, `min`, `oct`, `ord`, `pow`, `print`, `range`, `repr`, `reversed`, `round`, `sorted`, `str`, `sum`, `type`, `zip`.

This notebook covers everything else:

| Group | Functions |
|---|---|
| 1. Building collections | `list`, `tuple`, `dict`, `set`, `frozenset` |
| 2. Iterating | `iter`, `next`, `slice` |
| 3. Inspecting objects | `id`, `hash`, `dir`, `vars`, `help`, `callable`, `issubclass` |
| 4. Working with attributes | `getattr`, `setattr`, `hasattr`, `delattr` |
| 5. Classes | `object`, `property`, `classmethod`, `staticmethod`, `super` |
| 6. Files | `open` |
| 7. Binary data | `bytes`, `bytearray`, `memoryview` |
| 8. Running code from text | `eval`, `exec`, `compile`, `globals`, `locals` |
| 9. Async iteration | `aiter`, `anext` |
| 10. Debugging and the interpreter | `breakpoint`, `__import__`, `exit`, `quit`, `copyright`, `credits`, `license` |

## 1. Building collections

Called with no arguments, each gives an empty collection. Called with an iterable, it converts it.

In [ ]:
print(list("abc"), list(range(3)), list({"a": 1, "b": 2}))   # a dict gives its keys
print(tuple([1, 2, 3]), tuple("hi"))
print(set([3, 1, 3, 2, 1]))            # duplicates removed
print(frozenset([1, 2, 2]))
print(list(), tuple(), dict(), set())   # empty ones - note {} is an empty DICT, not a set


In [ ]:
# dict() has several forms
print(dict(name="John", age=25))                   # keyword arguments
print(dict([("name", "John"), ("age", 25)]))       # list of pairs
print(dict(zip(["name", "age"], ["John", 25])))    # two lists, with zip
print(dict.fromkeys(["a", "b", "c"], 0))           # same value for every key


In [ ]:
# set vs frozenset: frozenset can't be changed, so it can be a dict key or go inside a set
s = {1, 2}
s.add(3)
fs = frozenset({1, 2})
try:
    fs.add(3)
except AttributeError as e:
    print("AttributeError:", e)

routes = {frozenset({"Dallas", "Austin"}): 195}    # the order of cities doesn't matter
print(routes[frozenset({"Austin", "Dallas"})])


In [ ]:
# real world: convert to the right shape for the job
emails = ["a@x.com", "b@x.com", "a@x.com"]
print(len(set(emails)), "unique emails")            # set for uniqueness
point = tuple([3, 4])                               # tuple for fixed data that won't change
locations = {point: "home"}                         # ...which can be a dict key (lists can't)
print(locations)


## 2. Iterating: `iter`, `next`, `slice`

`iter` and `next` are covered fully in `02_iterator.ipynb`. Here are two extra forms.

In [ ]:
# next() with a default never raises StopIteration
it = iter([10, 20])
print(next(it), next(it), next(it, "done"))

# iter(function, sentinel): call the function until it returns the sentinel
import random
random.seed(1)
rolls = list(iter(lambda: random.randint(1, 6), 6))   # roll until you get a 6
print("rolls before a 6:", rolls)


In [ ]:
# slice() is the object behind the [start:stop:step] syntax
nums = list(range(10))
evens = slice(0, None, 2)
print(nums[evens], nums[0:None:2])    # same thing

# real world: name your slices when parsing fixed-width text
record = "John      Dallas    025"
NAME, CITY, AGE = slice(0, 10), slice(10, 20), slice(20, 23)
print(record[NAME].strip(), record[CITY].strip(), int(record[AGE]))


## 3. Inspecting objects

### `id()`: identity (the memory address in CPython)

In [ ]:
a = [1, 2]
b = a
c = [1, 2]
print(id(a) == id(b), a is b)   # True True  - same object
print(id(a) == id(c), a is c)   # False False - equal value, different object


### `hash()`: the number dicts and sets use to find items quickly

In [ ]:
print(hash(42), hash("hi") == hash("hi"), hash((1, 2)) == hash((1, 2)))

# only immutable values can be hashed, which is why lists can't be dict keys
try:
    hash([1, 2])
except TypeError as e:
    print("TypeError:", e)

# string hashes change every time Python starts (for security), so never save them


### `dir()`, `vars()`, `help()`: exploring objects

In [ ]:
# dir() lists the attributes and methods of anything
print([m for m in dir([]) if not m.startswith("_")])

class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

rex = Dog("Rex", 3)
print(vars(rex))      # an object's own data, as a dict


In [ ]:
# help() prints the documentation for anything
help(len)

# in Jupyter you can also type   len?   in a cell


### `callable()` and `issubclass()`

In [ ]:
print(callable(print), callable(len), callable(Dog))   # functions and classes: True
print(callable(42), callable("hi"))                    # values: False

# real world: accept either a value or a function that makes the value
def get(value_or_func):
    return value_or_func() if callable(value_or_func) else value_or_func

print(get(5), get(lambda: 5 * 2))


In [ ]:
# issubclass is like isinstance, but for classes instead of objects
print(issubclass(bool, int))                   # True - that's why True + True == 2
print(issubclass(ValueError, Exception))       # True
print(issubclass(int, (str, float)))           # False - a tuple checks several classes


## 4. Working with attributes by name

These let you use an attribute whose name is stored in a **string**, decided while the program runs.

In [ ]:
class Settings:
    theme = "dark"
    font_size = 12

s = Settings()
print(getattr(s, "theme"))                # same as s.theme
print(getattr(s, "language", "en"))       # default if it doesn't exist
print(hasattr(s, "font_size"), hasattr(s, "volume"))

setattr(s, "font_size", 14)               # same as s.font_size = 14
setattr(s, "volume", 80)                  # can create new attributes too
print(s.font_size, s.volume)

delattr(s, "volume")                      # same as del s.volume
print(hasattr(s, "volume"))


In [ ]:
# real world: apply settings from a config dict
config = {"theme": "light", "font_size": 16, "unknown_option": True}

for key, value in config.items():
    if hasattr(s, key):
        setattr(s, key, value)
    else:
        print(f"skipping unknown setting: {key}")
print(s.theme, s.font_size)


In [ ]:
# real world: pick a function by name, like a simple command menu
class Calculator:
    def add(self, a, b): return a + b
    def sub(self, a, b): return a - b

calc = Calculator()
for command in ["add 5 3", "sub 10 4", "mul 2 2"]:
    name, a, b = command.split()
    action = getattr(calc, name, None)
    print(command, "->", action(int(a), int(b)) if action else "unknown command")


## 5. Classes: `object`, `property`, `classmethod`, `staticmethod`, `super`

These are mostly used as **decorators** (`@name` above a method).

In [ ]:
# object is the base class of everything
print(isinstance(42, object), isinstance("hi", object), isinstance(Dog, object))
print(Dog.__mro__)        # Dog -> object

# object() makes a unique "sentinel" value, as in the iterator notebook
MISSING = object()
print(MISSING is MISSING, MISSING == object())


In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    # property: a method you read like an attribute (no parentheses)
    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    # setter: runs when you assign to it - good for validation
    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5 / 9

t = Temperature(100)
print(t.fahrenheit)     # 212.0 - no () needed
t.fahrenheit = 32
print(t.celsius)        # 0.0


In [ ]:
class Date:
    def __init__(self, year, month, day):
        self.year, self.month, self.day = year, month, day

    # classmethod: gets the CLASS (cls), usually used as another way to create objects
    @classmethod
    def from_string(cls, text):
        year, month, day = map(int, text.split("-"))
        return cls(year, month, day)

    # staticmethod: gets neither self nor cls - just a function kept inside the class
    @staticmethod
    def is_leap(year):
        return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)

d = Date.from_string("2026-10-02")
print(d.year, d.month, d.day)
print(Date.is_leap(2024), Date.is_leap(2026))


In [ ]:
# super(): call the parent class's version of a method
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} makes a sound"

class Cat(Animal):
    def __init__(self, name, indoor):
        super().__init__(name)      # let Animal set up the name
        self.indoor = indoor

    def speak(self):
        return super().speak() + ": meow"

c = Cat("Tom", indoor=True)
print(c.speak(), c.indoor)


## 6. Files: `open()`

Always use `with`: it closes the file automatically, even if an error happens.

| Mode | Meaning |
|---|---|
| `"r"` | read (default) - error if missing |
| `"w"` | write - **erases** the file first |
| `"a"` | append to the end |
| `"x"` | create - error if it already exists |
| add `"b"` | binary, e.g. `"rb"` for images |

In [ ]:
import os

path = "demo_notes.txt"

with open(path, "w", encoding="utf-8") as f:
    f.write("first line\n")
    f.write("second line\n")

with open(path, "a", encoding="utf-8") as f:
    print("third line", file=f)          # print() can write to a file too

with open(path, encoding="utf-8") as f:
    print(repr(f.read()))                # the whole file as one string

with open(path, encoding="utf-8") as f:
    for line in f:                       # a file is an iterator of lines - memory friendly
        print(line.rstrip())

os.remove(path)                          # clean up


In [ ]:
# reading a file that doesn't exist
try:
    open("no_such_file.txt")
except FileNotFoundError as e:
    print("FileNotFoundError:", e)


In [ ]:
# real world: save and load a list of scores as CSV text
scores = {"Amy": 92, "John": 85, "Raj": 78}
path = "demo_scores.csv"

with open(path, "w", encoding="utf-8") as f:
    f.write("name,score\n")
    for name, score in scores.items():
        f.write(f"{name},{score}\n")

with open(path, encoding="utf-8") as f:
    next(f)                                       # skip the header line
    loaded = {name: int(score) for name, score in (line.strip().split(",") for line in f)}
print(loaded)

os.remove(path)


## 7. Binary data: `bytes`, `bytearray`, `memoryview`

- `bytes`: an immutable sequence of numbers from 0 to 255 (like `str`, but for raw data).
- `bytearray`: the mutable version.
- `memoryview`: a window onto existing binary data, without copying it.

In [ ]:
b = bytes([72, 105, 33])
print(b, list(b), b[0])         # b'Hi!' [72, 105, 33] 72 - indexing gives an int
print(bytes(3))                 # b'\x00\x00\x00' - 3 zero bytes
print(bytes("café", "utf-8"))   # same as "café".encode()
print(bytes.fromhex("48 69"), b.hex())


In [ ]:
ba = bytearray(b"hello")
ba[0] = ord("H")                # can change in place - bytes can't
ba.extend(b"!!")
print(ba, bytes(ba))


In [ ]:
data = bytearray(b"abcdef")
view = memoryview(data)[2:5]    # no copy - shares memory with data
print(bytes(view))              # b'cde'
view[0] = ord("X")              # changing the view changes the original
print(data)                     # bytearray(b'abXdef')


## 8. Running code from text: `eval`, `exec`, `compile`, `globals`, `locals`

⚠️ **Never** pass text from users, files or the internet to `eval` or `exec`. It can run any code at all, including deleting files. For turning text into Python values safely, use `ast.literal_eval`.

In [ ]:
# eval: evaluate an EXPRESSION and return its value
print(eval("2 + 3 * 4"))
x = 10
print(eval("x * 2"))                          # can see your variables

# exec: run STATEMENTS (loops, assignments, defs) - returns None
exec("for i in range(3): print('exec', i)")

# compile: turn text into a code object once, to run many times
code = compile("n * n", "<formula>", "eval")
print([eval(code, {"n": n}) for n in range(5)])


In [ ]:
# the safe alternative for reading Python-style data from text
import ast
print(ast.literal_eval("[1, 2, {'a': 3}]"))
try:
    ast.literal_eval("__import__('os').getcwd()")
except ValueError:
    print("rejected: not a plain value")


In [ ]:
# globals(): a dict of all variables at the top level of the notebook
# locals():  a dict of the variables inside the current function
def show_locals(a, b):
    total = a + b
    return locals()

print(show_locals(2, 3))
print("x" in globals(), globals()["x"])


## 9. Async iteration: `aiter`, `anext`

The async versions of `iter` and `next`, for data that arrives over time (network messages, streams). Jupyter lets you use `await` directly in a cell.

In [ ]:
import asyncio

async def countdown(n):
    while n > 0:
        await asyncio.sleep(0.1)    # pretend we're waiting for data
        yield n
        n -= 1

async def main():
    it = aiter(countdown(3))
    print(await anext(it))              # 3
    print(await anext(it))              # 2
    print(await anext(it))              # 1
    print(await anext(it, "done"))      # default, like next()

    async for n in countdown(2):        # async for calls aiter/anext for you
        print("async for:", n)

await main()


## 10. Debugging and the interpreter

In [ ]:
# breakpoint() pauses the program and opens the debugger (pdb) at that line.
# In the debugger: n = next line, s = step into, p x = print x, c = continue, q = quit.
# Uncomment to try it:

def average(nums):
    total = sum(nums)
    # breakpoint()
    return total / len(nums)

print(average([1, 2, 3]))


In [ ]:
# __import__ imports a module by name from a string - normally you'd use importlib
math_module = __import__("math")
print(math_module.sqrt(16))

import importlib
print(importlib.import_module("math").pi)


In [ ]:
# exit, quit, copyright, credits, license are helpers for the interactive prompt
# (added by the "site" module), not for real programs. In scripts use sys.exit().
print(type(exit), type(credits))


## Checklist: every built-in, and where it is covered

In [ ]:
import builtins

numbers = "abs bin bool chr complex divmod float format hex int isinstance max min oct ord pow range round sum type"
strings = "all any ascii enumerate filter input len map print repr reversed sorted str zip"
this = """list tuple dict set frozenset iter next slice id hash dir vars help callable issubclass
getattr setattr hasattr delattr object property classmethod staticmethod super open bytes bytearray
memoryview eval exec compile globals locals aiter anext breakpoint __import__ exit quit copyright credits license"""

covered = set((numbers + " " + strings + " " + this).split())
all_builtins = {
    name for name in dir(builtins)
    if callable(getattr(builtins, name))
    and not (isinstance(getattr(builtins, name), type) and issubclass(getattr(builtins, name), BaseException))
    and (not name.startswith("_") or name == "__import__")
}
print(f"{len(all_builtins)} built-ins, {len(covered & all_builtins)} covered")
print("missing:", all_builtins - covered or "none")
